In [1]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import glob


In [2]:
fieldmap_file = "/Users/daisykalra/Desktop/UTK/muon-cooling/hfofo-frozen/g4bl-input/5period_25_300_1000_0.001/fieldmap.txt"
ztuple_pattern = "/Users/daisykalra/Desktop/UTK/muon-cooling/hfofo-frozen/g4bl-input/for009.txt"  

In [3]:
f = 0.325  # GHz
omega = 2 * np.pi * f
T = 1 / f

# =========================
# HELPERS
# =========================
def rf_mod(t, T):
    return ((t % T))

def wrap_360(phi):
    return np.mod(phi, 360)

def delta_phase_wrap(phi1, phi2):
    """Shortest difference in degrees"""
    return ((phi1 - phi2 + 180) % 360) - 180

def wrap_time(dt, T):
    return ((dt + T/2) % T) - T/2

# =========================
# LOAD FIELD MAP
# =========================
print("Loading field map...")

fcols = ["x","y","z","t","Bx","By","Bz","Ex","Ey","Ez"]
#field = pd.read_csv(fieldmap_file, sep=r"\s+", comment="#", names=fcols)

field = pd.read_csv(
    fieldmap_file,
    sep=r"\s+",
    comment="#",
    names=fcols,
    usecols=["z", "t", "Ez"]
)


Loading field map...


Python(1522,0x2078e49c0) malloc: Failed to allocate segment from range group - out of space


In [4]:
#print(field["z"].unique())

In [5]:
Ez_interp = {}
dEz_dt_interp = {}

for z_val in sorted(field["z"].unique()):
    slice_df = field[field["z"] == z_val]

    t_vals = slice_df["t"].values
    Ez_vals = slice_df["Ez"].values

    if len(t_vals) < 2:
        continue

    dEz_dt = np.gradient(Ez_vals, t_vals)

    Ez_interp[z_val] = interp1d(t_vals, Ez_vals, kind="cubic", fill_value="extrapolate")
    dEz_dt_interp[z_val] = interp1d(t_vals, dEz_dt, kind="cubic", fill_value="extrapolate")

print(f"Interpolators built for {len(Ez_interp)} z slices")

Interpolators built for 858 z slices


In [6]:
cols = [
    "IEVT","IPNUM","IPTYP","IPFLG","JSRG",
    "T","X","Y","Z","Px","Py","Pz",
    "Bx","By","Bz","Weight","Ex","Ey","Ez",
    "SARC","POLx","POLy","POLz"
]

#df = pd.read_csv(ztuple_pattern, sep=r"\s+", comment="#", names=cols)

df = pd.read_csv(
    ztuple_pattern,
    sep=r"\s+",
    comment="#",
    names=cols,
    usecols=["T", "Z", "Ez"]  
)

# ----------------------------
# Ensure consistent units: particle Z in mm
# ----------------------------
df["Z_mm"] = df["Z"] * 1000   # meters -> mm
particle_z = df["Z_mm"].values
field_z_list = np.sort(np.array(list(Ez_interp.keys())))  # already in mm

# ----------------------------
# Map each particle Z to nearest field slice (memory-efficient)
# ----------------------------
idx = np.searchsorted(field_z_list, particle_z)
idx[idx == len(field_z_list)] = len(field_z_list) - 1

left = np.clip(idx - 1, 0, len(field_z_list) - 1)
right = idx

nearest_z = np.where(
    np.abs(particle_z - field_z_list[left]) <= np.abs(particle_z - field_z_list[right]),
    field_z_list[left],
    field_z_list[right]
)

df["z_key"] = nearest_z

# ----------------------------
# Loop over field slices and compute phase
# ----------------------------
results = []

for z_key in field_z_list:

    df_z = df[df["z_key"] == z_key]

    if df_z.empty:
        continue

    #print(f"Processing field z = {z_key} with {len(df_z)} particles")

    t_particles = df_z["T"].values

    # TRUE PHASE (FIELD)
    Ez_vals = Ez_interp[z_key](t_particles)
    dEz_vals = dEz_dt_interp[z_key](t_particles)
    phi_field = np.degrees(np.arctan2(omega * Ez_vals, dEz_vals))
    phi_field = wrap_360(phi_field)

    # OLD PHASE (TIMING)
    t_mod = rf_mod(t_particles, T)
    phi_old = np.degrees(omega * t_mod)
    phi_old = wrap_360(phi_old)

    # PHASE DIFFERENCE
    dphi = delta_phase_wrap(phi_field, phi_old)

    # TIME OFFSET
    dt = np.nanmean(dphi / omega)
    dt = wrap_time(dt, T)

    results.append((z_key, dt))

print("Phase calculation completed for all field slices.")

Phase calculation completed for all field slices.


In [7]:
# =========================
# FINAL RESULTS
# =========================
df_out = pd.DataFrame(results, columns=["z", "delta_t"])
df_out = df_out.sort_values("z")

print("\n=== OFFSETS vs Z ===")
print(df_out)



=== OFFSETS vs Z ===
         z   delta_t
0     -425 -1.207234
1     -400 -1.207234
2     -375 -1.207234
3     -350 -1.207234
4     -325 -1.207234
..     ...       ...
853  20900  1.296946
854  20925  1.296946
855  20950  1.296946
856  20975  1.296946
857  21000  1.296936

[858 rows x 2 columns]


In [10]:
z_query = 4000  # mm (example)

offset = df_out[df_out["z"] == z_query]["delta_t"]

print(offset)

177   -0.000023
Name: delta_t, dtype: float64


In [11]:
# =========================
# GROUP INTO toffs
# =========================
groups = {
    "toffs0": (-425, 576),
    "toffs1": (1000, 6300),
    "toffs2": (6300, 42000),
    "toffs3": (42000, 84000),
    "toffs4": (84000, 126000),
    "toffs5": (126000, 170000)
}

print("\n=== GROUP OFFSETS ===")

for key, (zmin, zmax) in groups.items():

    mask = (df_out["z"] >= zmin) & (df_out["z"] < zmax)
    vals = df_out[mask]["delta_t"].values

    if len(vals) == 0:
        continue

    mean_dt = wrap_time(np.mean(vals), T)

    print(f"{key} = {mean_dt:.5f} ns")


=== GROUP OFFSETS ===
toffs0 = -0.53376 ns
toffs1 = -0.00888 ns
toffs2 = -0.12755 ns


# deprecated


In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

# -----------------------------
# Constants
# -----------------------------
f = 0.325  # GHz
omega = 2 * np.pi * f  # rad/ns

# -----------------------------
# Load particle data
# -----------------------------
cols = ["x","y","z","Px","Py","Pz","t","PDG","EventID","TrackID","ParentID","Weight"]
particles = pd.read_csv("out.txt", comment="#", delim_whitespace=True, names=cols)

t_particles = particles["t"].values
z_particles = particles["z"].values

# Assume all particles at same detector z
z_det = np.mean(z_particles)

# -----------------------------
# Load fieldmap
# -----------------------------
fcols = ["x","y","z","t","Bx","By","Bz","Ex","Ey","Ez"]
field = pd.read_csv("fieldmap.txt", comment="#", delim_whitespace=True, names=fcols)

# -----------------------------
# Select closest z slice
# -----------------------------
z_unique = field["z"].unique()
z_field = z_unique[np.argmin(np.abs(z_unique - z_det))]

field_slice = field[field["z"] == z_field].copy()

t_field = field_slice["t"].values
Ez_field = field_slice["Ez"].values

# -----------------------------
# Compute dEz/dt
# -----------------------------
dEz_dt = np.gradient(Ez_field, t_field)

# -----------------------------
# Interpolators
# -----------------------------
Ez_interp = interp1d(t_field, Ez_field, kind='cubic', fill_value="extrapolate")
dEz_dt_interp = interp1d(t_field, dEz_dt, kind='cubic', fill_value="extrapolate")

# -----------------------------
# Compute RF phase (correct way)
# -----------------------------
phase_field = []

for t in t_particles:
    Ez = Ez_interp(t)
    dEz = dEz_dt_interp(t)

    # phase using atan2 (robust!)
    phi = np.arctan2(Ez, dEz / omega)   # radians
    phase_field.append(np.degrees(phi))

phase_field = np.array(phase_field)

# -----------------------------
# OLD phase (your method)
# -----------------------------
T = 1 / f

def rf_mod(t, T):
    return ((t + T/2) % T) - T/2

phase_old = np.degrees(omega * rf_mod(t_particles, T))

# -----------------------------
# Compare
# -----------------------------
delta_phase = phase_field - phase_old
delta_t = np.mean(delta_phase / omega)

print("Mean phase (old):", np.mean(phase_old))
print("Mean phase (field):", np.mean(phase_field))
print("Suggested time shift (ns):", delta_t)

# -----------------------------
# Plot
# -----------------------------
plt.hist(phase_old, bins=50, alpha=0.5, label="Old phase")
plt.hist(phase_field, bins=50, alpha=0.5, label="Field phase")
plt.xlabel("Phase (deg)")
plt.ylabel("Counts")
plt.legend()
plt.show()